# Детекция меланомы | Часть 2: Поиск гиперпараметров и обучение ResNet-50

| | |
|---|---|
| **Датасет** | ISIC 2020 (33 126 изображений, менее 2% меланом) |
| **Задача** | Бинарная классификация: меланома / доброкачественное |
| **Архитектура** | ResNet-50 pretrained ImageNet, full fine-tuning |
| **Главная метрика** | AUC-PR (Average Precision) |
| **Вспомогательные** | F1, Recall, Precision, Specificity, Balanced Accuracy |

---

## Структура ноутбука

Поиск лучших гиперпараметров ведётся автоматически через Optuna, затем лучшие конфигурации дообучаются на большем числе эпох.

| Блок | Что делается | Эпох |
|---|---|---|
| **1. Optuna** | Автоматический поиск гиперпараметров: оптимизатор, планировщик, lr, weight decay, dropout, label smoothing | 10 × 20 trials |
| **2. Топ-3** | Три лучшие конфигурации обучаются полноценно, сохраняются веса и метрики | 20 |
| **3. Финал** | Лучшая модель дообучается с early stopping (patience=7) | 50 |

## 1. Конфигурация и импорты

In [1]:
import os
import json
import random
import shutil
import time
import copy

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
import torchvision.models as models

from sklearn.metrics import (
    average_precision_score,
    f1_score,
    recall_score,
    precision_score,
    balanced_accuracy_score,
    confusion_matrix,
)

import optuna
from optuna.pruners import MedianPruner

from torch.cuda.amp import autocast, GradScaler

import IPython.display as ipd

In [2]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

In [3]:
# Пути к данным
IMAGE_DIR = "/kaggle/input/datasets/cdeotte/jpeg-melanoma-256x256/train"
TRAIN_CSV  = '/kaggle/input/datasets/daryanikitina/splits/split_train.csv'
VAL_CSV    = '/kaggle/input/datasets/daryanikitina/splits/split_val.csv'
TEST_CSV   = '/kaggle/input/datasets/daryanikitina/splits/split_test.csv'
OUTPUT_DIR = '/kaggle/working'

# DataLoader
BATCH_SIZE  = 32
NUM_WORKERS = 2

In [4]:
# Optuna: база данных для сохранения trials между сессиями
STUDY_DB_PATH = '/kaggle/working/optuna_resnet.db'
STUDY_NAME    = 'resnet50_stage1'

PREV_DB_PATH  = '/kaggle/input/datasets/daryanikitina/optuna-resnet/optuna_resnet (1).db'

# Блок 1: Optuna
N_TRIALS_TOTAL    = 20
OPTUNA_EPOCHS     = 10
# Останавливаю за 1 час до конца сессии Kaggle
TIME_BUDGET_HOURS = 11.0

# Блок 2: топ-3 на полные 20 эпох
NUM_EPOCHS_BLOCK2 = 20
TOP_K             = 3

# Блок 3: дообучение лучшей модели
NUM_EPOCHS_BLOCK3       = 30  # ещё 30 эпох поверх блока 2 (вместе 50)
EARLY_STOPPING_PATIENCE = 7

In [5]:
# Воспроизводимость: seed=42 везде
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ['PYTHONHASHSEED'] = str(SEED)

In [6]:
scaler = GradScaler()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Устройство: {DEVICE}")

Устройство: cuda


## 2. Загрузка данных

In [7]:
class MelanomaDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, row["image_name"] + ".jpg")
        image    = Image.open(img_path).convert("RGB")  # защита от RGBA/grayscale
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(row["target"], dtype=torch.float32)  # float32 для BCEWithLogitsLoss
        return image, label

In [8]:
# Статистики нормализации ImageNet - модель обучена на этих значениях, поэтому входные данные нужно привести к тому же масштабу
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

In [9]:
# Для обучения аугментация из первой части - базовая геометрическая
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),    
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Для валидации и теста нет аугментации
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

In [10]:
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

In [11]:
train_dataset = MelanomaDataset(train_df, IMAGE_DIR, transform=train_transform)
val_dataset   = MelanomaDataset(val_df,   IMAGE_DIR, transform=val_transform)
test_dataset  = MelanomaDataset(test_df,  IMAGE_DIR, transform=val_transform)

In [12]:
# Balanced batch
n_class0 = int((train_df['target'] == 0).sum())
n_class1 = int((train_df['target'] == 1).sum())
weight_class0  = 1.0 / n_class0
weight_class1  = 1.0 / n_class1
sample_weights = train_df['target'].map({0: weight_class0, 1: weight_class1}).values

sampler = WeightedRandomSampler(
    weights     = torch.FloatTensor(sample_weights),
    num_samples = len(train_dataset),
    replacement = True,  # с возвратом
)

In [13]:
train_loader = DataLoader(
    train_dataset,
    batch_size  = BATCH_SIZE,
    sampler     = sampler,
    num_workers = NUM_WORKERS,
    pin_memory  = False,
)
val_loader = DataLoader(
    val_dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    num_workers = NUM_WORKERS,
    pin_memory  = False,
)
test_loader = DataLoader(
    test_dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    num_workers = NUM_WORKERS,
    pin_memory  = False,
)

print('\nDataLoader созданы:')
print('  Батчей в train:', len(train_loader))
print('  Батчей в val:  ', len(val_loader))
print('  Батчей в test: ', len(test_loader))


DataLoader созданы:
  Батчей в train: 777
  Батчей в val:   130
  Батчей в test:  130


## 3. Вспомогательные функции

In [14]:
class LabelSmoothingBCELoss(nn.Module):
    """
    BCEWithLogitsLoss с label smoothing (сглаживание меток).
    Стандартный BCE учит модель быть абсолютно уверенной, что ведёт к переобучению. Label smoothing 'размывает' жёсткие метки {0, 1}.
    В медицинских задачах метки могут быть зашумлены (спорные случаи на границе), поэтому небольшое сглаживание часто улучшает обобщение на новых данных.
    """
    def __init__(self, label_smoothing=0.0):
        super().__init__()
        self.ls  = label_smoothing
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        # Сдвигаю метки от краёв шкалы [0,1] ближе к центру
        soft_targets = targets * (1.0 - self.ls) + 0.5 * self.ls
        return self.bce(logits, soft_targets)

In [15]:
def make_resnet50(dropout=0.0):
    """
    Создаёт ResNet-50 с предобучением ImageNet, заменяет последний fc-слой.

    Использую предобученные веса, так как первые слои ResNet обучены находить края, текстуры, формы - универсальные
    признаки, полезные для любых изображений. Fine-tuning переиспользует их, что быстрее и лучше, чем обучение с нуля при ограниченном датасете.

    Dropout перед fc-слоем - случайно отключает нейроны во время обучения, не давая модели полагаться на конкретные признаки - переобучаться.
    """
    try:
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    except AttributeError:
        model = models.resnet50(pretrained=True)

    in_features = model.fc.in_features
    if dropout > 0.0:
        model.fc = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_features, 1),
        )
    else:
        model.fc = nn.Linear(in_features, 1)
    return model

In [16]:
def compute_metrics(model, loader, threshold=None):
    """
    Прогоняет модель через весь DataLoader, считает все метрики качества.

    threshold=None  -> ищет лучший порог по F1 на переданных данных (для val).
    threshold=float -> применяет фиксированный порог без поиска (для test).
    """
    model.eval()
    all_probs  = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            logits = model(images).squeeze(1)
            probs  = torch.sigmoid(logits).cpu().numpy()
            all_probs.extend(probs.tolist())
            all_labels.extend(labels.numpy().tolist())

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)

    # Защита от вырожденного случая (все метки одного класса - бывает в начале обучения)
    if len(np.unique(all_labels)) < 2:
        auc_pr = 0.0
    else:
        # AUC-PR не зависит от порога - суммирует качество по всем возможным порогам
        auc_pr = float(average_precision_score(all_labels, all_probs))

    # Поиск порога, максимизирующего F1 - перебираю все 99 вариантов
    if threshold is None:
        best_f1  = -1.0
        best_thr = 0.5
        for thr in np.arange(0.01, 1.00, 0.01):
            preds_thr = (all_probs >= thr).astype(int)
            f1_thr    = f1_score(all_labels, preds_thr, zero_division=0.0)
            if f1_thr > best_f1:
                best_f1  = f1_thr
                best_thr = float(thr)
        threshold = best_thr

    preds = (all_probs >= threshold).astype(int)

    f1      = float(f1_score(all_labels, preds, zero_division=0.0))
    recall  = float(recall_score(all_labels, preds, zero_division=0.0))
    prec    = float(precision_score(all_labels, preds, zero_division=0.0))
    bal_acc = float(balanced_accuracy_score(all_labels, preds))

    cm = confusion_matrix(all_labels, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    specificity = float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0

    return {
        'auc_pr':            auc_pr,
        'f1':                f1,
        'recall':            recall,
        'precision':         prec,
        'specificity':       specificity,
        'balanced_accuracy': bal_acc,
        'threshold':         threshold,
    }

In [17]:
def make_optimizer(params, model):
    """
    Создаёт оптимизатор по словарю params (от Optuna или сохранённому конфигу).

    Adam - стандарт; адаптивный lr для каждого параметра, хорошо работает 'из коробки'.
    AdamW - как Adam, но weight decay применяется правильно (не через lr, а отдельно). Обычно лучше Adam при fine-tuning предобученных моделей.
    SGD+momentum - классика; может лучше обобщать, но требует аккуратной настройки lr.
    """
    name = params['optimizer']
    lr   = params['lr']
    wd   = params['weight_decay']
    if name == 'Adam':
        return optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    elif name == 'AdamW':
        return optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    else:  # SGD
        return optim.SGD(model.parameters(), lr=lr, weight_decay=wd, momentum=0.9)

In [18]:
def train_one_epoch(model, loader, optimizer, criterion, scheduler=None, scaler=None):
    """
    Обучает модель одну эпоху. Возвращает средний loss по батчам.

    OneCycleLR делает scheduler.step() после КАЖДОГО батча: lr-цикл рассчитан на точное число шагов = epochs x batches_per_epoch. 
    Все остальные schedulers делают шаг 1 раз в конце эпохи.
    """
    model.train()
    total_loss = 0.0
    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        optimizer.zero_grad()
        
        # Mixed precision: вычисления в float16 вместо float32 - в 2 раза быстрее
        with autocast():
            logits = model(images).squeeze(1)
            loss   = criterion(logits, labels)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        if scheduler is not None and isinstance(scheduler, optim.lr_scheduler.OneCycleLR):
            scheduler.step()
        del images, labels, logits, loss
    
    if scheduler is not None and not isinstance(scheduler, optim.lr_scheduler.OneCycleLR):
        scheduler.step()
    return total_loss / len(loader)

In [19]:
def make_scheduler(params, optimizer, n_epochs, steps_per_epoch):
    """
    Создаёт scheduler по словарю params.

    StepLR: снижает lr в 2 раза каждые 3 эпохи. Просто и предсказуемо.
    CosineAnnealingLR: плавно снижает lr по косинусу за T_max эпох, затем возвращается к максимуму. Помогает 'перепрыгивать' локальные минимумы.
    OneCycleLR: один цикл - lr сначала растёт до max_lr, потом падает. Часто сходится быстрее, но нужно заранее знать число эпох.

    n_epochs и steps_per_epoch используются только для OneCycleLR.
    """
    name = params['scheduler']
    lr   = params['lr']
    if name == 'StepLR':
        return optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)
    elif name == 'CosineAnnealingLR':
        return optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
    else:  # OneCycleLR
        return optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr          = lr,
            epochs          = n_epochs,
            steps_per_epoch = steps_per_epoch,
        )

In [20]:
def save_results(results, path):
    """
    Сохраняет список словарей в JSON.
    Ключ 'model' пропускается, так как это объект модели + показывает кликабельную ссылку для скачивания файла.
    """
    clean = [{k: v for k, v in r.items() if k != 'model'} for r in results]
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(clean, f, indent=2, ensure_ascii=False)
    print('Сохранено:', path)
    ipd.display(ipd.FileLink(path))

In [21]:
def display_db_link(path):
    """Выводит кликабельную ссылку для скачивания базы данных Optuna."""
    print('База данных Optuna:', path)
    ipd.display(ipd.FileLink(path))

---
## 4. Автоматический поиск гиперпараметров - Optuna

**Что ищу:** optimizer, scheduler, lr, weight\_decay, dropout, label\_smoothing  
**Как:** каждый trial - одно обучение на `OPTUNA_EPOCHS=10` эпох  
**Метрика trial-а:** среднее val AUC-PR за последние 3 эпохи  
**MedianPruner:** срезает trial, если его метрика хуже медианы других на той же эпохе  
**SQLite:** trials сохраняются в `.db` - можно продолжить из следующей сессии Kaggle

In [22]:
# Настройка SQLite-хранилища и создание study

if PREV_DB_PATH is not None and os.path.exists(PREV_DB_PATH):
    shutil.copy(PREV_DB_PATH, STUDY_DB_PATH)
    print('Скопирована база из предыдущей сессии:', PREV_DB_PATH)
else:
    print('Начинаю новое исследование Optuna с нуля')

storage_url = 'sqlite:///' + STUDY_DB_PATH

# MedianPruner срезает trial, если его текущая метрика хуже медианы всех других
# n_startup_trials=5: первые 5 trials не трогаю - нужна база для сравнения
# n_warmup_steps=4: первые 4 эпохи каждого trial тоже не обрезаю - модель ещё учится
pruner = MedianPruner(n_startup_trials=5, n_warmup_steps=4)

study = optuna.create_study(
    study_name     = STUDY_NAME,
    storage        = storage_url,
    direction      = 'maximize',
    pruner         = pruner,
    load_if_exists = True,
)

already_done = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
print('Уже завершено trials:', len(already_done))

Скопирована база из предыдущей сессии: /kaggle/input/datasets/daryanikitina/optuna-resnet/optuna_resnet (1).db


[I 2026-06-16 13:32:54,431] Using an existing study with name 'resnet50_stage1' instead of creating a new one.


Уже завершено trials: 11


In [23]:
# Objective function и TimeBudgetCallback

def optuna_objective(trial):
    """
    Одна итерация Optuna: пробует набор параметров, обучает модель OPTUNA_EPOCHS эпох,
    возвращает среднее val AUC-PR за последние 3 эпохи как оценку качества.
    """
    # Optuna предлагает значения из заданных диапазонов
    opt_name     = trial.suggest_categorical('optimizer',       ['Adam', 'AdamW', 'SGD'])
    sched_name   = trial.suggest_categorical('scheduler',       ['StepLR', 'CosineAnnealingLR', 'OneCycleLR'])
    lr           = trial.suggest_float('lr',            1e-5, 1e-2, log=True)
    weight_decay = trial.suggest_float('weight_decay',  1e-5, 1e-2, log=True)
    dropout      = trial.suggest_float('dropout',       0.0,  0.5)
    label_smooth = trial.suggest_float('label_smoothing', 0.0, 0.2)

    params = {
        'optimizer':       opt_name,
        'scheduler':       sched_name,
        'lr':              lr,
        'weight_decay':    weight_decay,
        'dropout':         dropout,
        'label_smoothing': label_smooth,
    }

    trial_model = make_resnet50(dropout=dropout).to(DEVICE)
    trial_opt   = make_optimizer(params, trial_model)
    trial_sched = make_scheduler(params, trial_opt, OPTUNA_EPOCHS, len(train_loader))
    criterion   = LabelSmoothingBCELoss(label_smoothing=label_smooth)

    val_auc_pr_history = []

    scaler = GradScaler()

    for epoch in range(OPTUNA_EPOCHS):
        train_one_epoch(trial_model, train_loader, trial_opt, criterion, trial_sched, scaler=scaler)
        metrics    = compute_metrics(trial_model, val_loader)
        val_auc_pr = metrics['auc_pr']
        val_auc_pr_history.append(val_auc_pr)

        trial.report(val_auc_pr, epoch)

        # Принудительный прунинг: если после 5-й эпохи среднее AUC-PR < 0.05, модель почти не лучше случайного классификатора
        if epoch >= 4:
            if float(np.mean(val_auc_pr_history)) < 0.05:
                del trial_model, trial_opt, trial_sched, criterion
                torch.cuda.empty_cache()
                raise optuna.TrialPruned()

        if trial.should_prune():
            del trial_model, trial_opt, trial_sched, criterion
            torch.cuda.empty_cache()
            raise optuna.TrialPruned()

    result_value = float(np.mean(val_auc_pr_history[-3:]))

    # Освобождаем GPU-память
    del trial_model, trial_opt, trial_sched, criterion
    torch.cuda.empty_cache()

    return result_value


class TimeBudgetCallback:
    """
    Останавливает Optuna, если исчерпан временной бюджет.
    Kaggle автоматически убивает сессию через 12 часов, поэтому нужно успеть сохраниться.
    """
    def __init__(self, hours):
        # момент времени, после которого нужно остановиться
        self.deadline = time.time() + hours * 3600.0

    def __call__(self, study, trial):
        if time.time() > self.deadline:
            print('Временной бюджет исчерпан - останавливаю Optuna.')
            study.stop()

In [31]:
# Запуск оптимизации + вывод результатов

time_callback = TimeBudgetCallback(TIME_BUDGET_HOURS)

already_done = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
remaining = max(0, N_TRIALS_TOTAL - already_done)
print('Уже завершено: {} | Осталось запустить: {}'.format(already_done, remaining))

study.optimize(
    optuna_objective,
    n_trials          = remaining,
    callbacks         = [time_callback],
    show_progress_bar = True,
)

# Итоги
completed_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
pruned_trials    = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]

print('\n' + '='*70)
print('РЕЗУЛЬТАТЫ OPTUNA')
print('='*70)
print('Завершено: {} | Обрезано: {}'.format(len(completed_trials), len(pruned_trials)))

print('\nВсе завершённые trials (по убыванию метрики):')
for t in sorted(completed_trials, key=lambda x: x.value, reverse=True):
    print('  Trial {:3d} | val_auc_pr={:.4f} | {}'.format(t.number, t.value, t.params))

best = study.best_trial
print('\nЛучший trial: #{}'.format(best.number))
print('  Значение: {:.4f}'.format(best.value))
print('  Параметры:')
for k, v in best.params.items():
    print('    {}: {}'.format(k, v))

display_db_link(STUDY_DB_PATH)

Уже завершено: 1 | Осталось запустить: 19


  0%|          | 0/19 [00:00<?, ?it/s]

/tmp/ipykernel_58/796165264.py:32: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_58/3094333196.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_58/3094333196.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_58/3094333196.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_58/3094333196.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_58/3094333196.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. P

[I 2026-06-16 00:21:32,246] Trial 4 finished with value: 0.21985361960595654 and parameters: {'optimizer': 'SGD', 'scheduler': 'StepLR', 'lr': 0.00548907301706208, 'weight_decay': 7.015598832542148e-05, 'dropout': 0.39906177489561057, 'label_smoothing': 0.10191994318156339}. Best is trial 4 with value: 0.21985361960595654.


/tmp/ipykernel_58/796165264.py:32: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_58/3094333196.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_58/3094333196.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_58/3094333196.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_58/3094333196.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_58/3094333196.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. P

...

/tmp/ipykernel_58/796165264.py:32: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_58/3094333196.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_58/3094333196.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_58/3094333196.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_58/3094333196.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_58/3094333196.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. P

[I 2026-06-16 04:56:17,001] Trial 21 pruned. 


/tmp/ipykernel_58/796165264.py:32: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_58/3094333196.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_58/3094333196.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_58/3094333196.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_58/3094333196.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_58/3094333196.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. P

[I 2026-06-16 05:06:52,796] Trial 22 pruned. 

РЕЗУЛЬТАТЫ OPTUNA
Завершено: 11 | Обрезано: 9

Все завершённые trials (по убыванию метрики):
  Trial   6 | val_auc_pr=0.2679 | {'optimizer': 'Adam', 'scheduler': 'CosineAnnealingLR', 'lr': 1.2733617885633793e-05, 'weight_decay': 0.001393578530649149, 'dropout': 0.16977102154288687, 'label_smoothing': 0.11227438607129932}
  Trial  15 | val_auc_pr=0.2610 | {'optimizer': 'SGD', 'scheduler': 'CosineAnnealingLR', 'lr': 0.002589621870800469, 'weight_decay': 0.0009194938792190307, 'dropout': 0.2824974186827375, 'label_smoothing': 0.04630851024062921}
  Trial   9 | val_auc_pr=0.2577 | {'optimizer': 'SGD', 'scheduler': 'CosineAnnealingLR', 'lr': 0.0016038791115402966, 'weight_decay': 0.0003882387971540929, 'dropout': 0.27270717062691385, 'label_smoothing': 0.029551498175417336}
  Trial  16 | val_auc_pr=0.2535 | {'optimizer': 'Adam', 'scheduler': 'CosineAnnealingLR', 'lr': 0.00010735278851810454, 'weight_decay': 0.0042862341859204525, 'dropout': 0.0

/kaggle/working/optuna_resnet.db

In [32]:
rows = []
for t in sorted(completed_trials, key=lambda x: x.value, reverse=True):
    row = {
        'Rank':         None,
        'Trial':        t.number,
        'AUC-PR (mean last 3)': round(t.value, 4),
        'optimizer':    t.params['optimizer'],
        'scheduler':    t.params['scheduler'],
        'lr':           round(t.params['lr'], 6),
        'weight_decay': round(t.params['weight_decay'], 6),
        'dropout':      round(t.params['dropout'], 3),
        'label_smooth': round(t.params['label_smoothing'], 3),
    }
    rows.append(row)

summary_df = pd.DataFrame(rows)
summary_df['Rank'] = range(1, len(summary_df) + 1)
summary_df

,Rank,Trial,AUC-PR (mean last 3),optimizer,scheduler,lr,weight_decay,dropout,label_smooth
0,1,6,0.2679,Adam,CosineAnnealingLR,0.000013,0.001394,0.170,0.112
1,2,15,0.2610,SGD,CosineAnnealingLR,0.002590,0.000919,0.282,0.046
2,3,9,0.2577,SGD,CosineAnnealingLR,0.001604,0.000388,0.273,0.030
3,4,16,0.2535,Adam,CosineAnnealingLR,0.000107,0.004286,0.016,0.139
4,5,17,0.2439,SGD,CosineAnnealingLR,0.002142,0.001877,0.199,0.053
5,6,18,0.2367,Adam,CosineAnnealingLR,0.000059,0.000567,0.305,0.118
6,7,5,0.2253,AdamW,StepLR,0.000207,0.000235,0.125,0.109
7,8,4,0.2199,SGD,StepLR,0.005489,0.000070,0.399,0.102
8,9,13,0.2184,Adam,CosineAnnealingLR,0.000074,0.000010,0.011,0.002
9,10,0,0.1962,Adam,CosineAnnealingLR,0.000527,0.000147,0.382,0.047


---
## 5. Топ-3 конфигурации оптуны на 20 эпох

Беру 3 лучших завершённых (не обрезанных) trial из Optuna. Каждый обучаю на `NUM_EPOCHS_BLOCK2=20` эпох с нуля. После каждой модели сохраняю `.pth` (веса + optimizer + scheduler) и `.json` с метриками. Все метрики выводятся на каждой эпохе.  
**Итоговая метрика модели:** среднее val AUC-PR за последние 3 эпохи.  

In [24]:
# Выбираю топ-K завершённых trials
completed_trials = sorted(
    [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE],
    key=lambda t: t.value,
    reverse=True,
)

if len(completed_trials) == 0:
    raise RuntimeError('Нет завершённых trials.')

top_k_trials = completed_trials[:TOP_K]
print('Топ-{} конфигураций для обучения:'.format(TOP_K))
for rank, t in enumerate(top_k_trials):
    print('  Rank {}: Trial #{} | optuna_val_auc_pr={:.4f}'.format(rank + 1, t.number, t.value))

# Загружаю уже сохранённые результаты (защита от перезапуска)
block2_json_path = os.path.join(OUTPUT_DIR, 'block2_results.json')

if os.path.exists(block2_json_path):
    with open(block2_json_path, 'r', encoding='utf-8') as f:
        block2_results = json.load(f)
    print('\nЗагружены результаты предыдущего запуска: {} моделей'.format(len(block2_results)))
else:
    block2_results = []

Топ-3 конфигураций для обучения:
  Rank 1: Trial #6 | optuna_val_auc_pr=0.2679
  Rank 2: Trial #15 | optuna_val_auc_pr=0.2610
  Rank 3: Trial #9 | optuna_val_auc_pr=0.2577


In [25]:
 scaler = torch.cuda.amp.GradScaler()

# Обучаю каждую из топ-3 моделей
for rank, trial in enumerate(top_k_trials):
    trial_key  = 'trial_{}'.format(trial.number)
    rank_label = rank + 1  # 1, 2, 3

    # Проверяю, обучена ли уже эта модель в предыдущем запуске
    if any(r.get('trial_key') == trial_key for r in block2_results):
        print('\nRank {} (Trial #{}): уже обучена - пропускаю.'.format(
            rank_label, trial.number))
        continue

    params = trial.params
    print('\n' + '='*70)
    print('RANK {} | Trial #{} | Optuna val_auc_pr={:.4f}'.format(
        rank_label, trial.number, trial.value))
    print('Параметры: {}'.format(params))
    print('='*70)

    # Создаю модель, оптимизатор, планировщик, функцию потерь
    model     = make_resnet50(dropout=params['dropout']).to(DEVICE)
    optimizer = make_optimizer(params, model)
    scheduler = make_scheduler(params, optimizer, NUM_EPOCHS_BLOCK2, len(train_loader))
    criterion = LabelSmoothingBCELoss(label_smoothing=params['label_smoothing'])

    epoch_history      = []   # полная история всех метрик по эпохам
    val_auc_pr_history = []   # только AUC-PR для скользящего среднего

    for epoch in range(NUM_EPOCHS_BLOCK2):
        epoch_num = epoch + 1

        # Обучаю одну эпоху
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scheduler, scaler=scaler)

        # Все метрики на val (порог подбирается отдельно на каждой эпохе по F1)
        val_metrics = compute_metrics(model, val_loader, threshold=None)
        val_auc_pr_history.append(val_metrics['auc_pr'])

        # Скользящее среднее за последние 3 эпохи
        if epoch >= 2:
            sliding3     = float(np.mean(val_auc_pr_history[-3:]))
            sliding3_str = ' | sliding3={:.4f}'.format(sliding3)
        else:
            sliding3     = None
            sliding3_str = ''

        epoch_history.append({
            'epoch':            epoch_num,
            'train_loss':       round(train_loss, 6),
            'val_auc_pr':       round(val_metrics['auc_pr'], 6),
            'val_f1':           round(val_metrics['f1'], 6),
            'val_recall':       round(val_metrics['recall'], 6),
            'val_precision':    round(val_metrics['precision'], 6),
            'val_specificity':  round(val_metrics['specificity'], 6),
            'val_balanced_acc': round(val_metrics['balanced_accuracy'], 6),
            'val_threshold':    round(val_metrics['threshold'], 4),
            'sliding_mean_3':   round(sliding3, 6) if sliding3 is not None else None,
        })

        # Вывод прогресса - все метрики на каждой эпохе
        print(
            'Ep {:2d}/{} | loss={:.4f} | AUC-PR={:.4f} | F1={:.4f} | '
            'Rec={:.4f} | Prec={:.4f} | Spec={:.4f} | BalAcc={:.4f} | '
            'thr={:.2f}{}'.format(
                epoch_num, NUM_EPOCHS_BLOCK2,
                train_loss,
                val_metrics['auc_pr'],
                val_metrics['f1'],
                val_metrics['recall'],
                val_metrics['precision'],
                val_metrics['specificity'],
                val_metrics['balanced_accuracy'],
                val_metrics['threshold'],
                sliding3_str,
            )
        )

    # Ищу лучший порог на val - он будет применён к test
    final_val_metrics  = compute_metrics(model, val_loader, threshold=None)
    final_test_metrics = compute_metrics(model, test_loader,
                                         threshold=final_val_metrics['threshold'])

    mean_last3 = float(np.mean(val_auc_pr_history[-3:]))

    print('\nФинальные метрики (порог найден на val, применён к test):')
    print('  Val  | AUC-PR={:.4f} | F1={:.4f} | Rec={:.4f} | '
          'Prec={:.4f} | Spec={:.4f} | BalAcc={:.4f} | thr={:.2f}'.format(
        final_val_metrics['auc_pr'], final_val_metrics['f1'],
        final_val_metrics['recall'], final_val_metrics['precision'],
        final_val_metrics['specificity'], final_val_metrics['balanced_accuracy'],
        final_val_metrics['threshold']))
    print('  Test | AUC-PR={:.4f} | F1={:.4f} | Rec={:.4f} | '
          'Prec={:.4f} | Spec={:.4f} | BalAcc={:.4f}'.format(
        final_test_metrics['auc_pr'], final_test_metrics['f1'],
        final_test_metrics['recall'], final_test_metrics['precision'],
        final_test_metrics['specificity'], final_test_metrics['balanced_accuracy']))
    print('  Итоговая метрика (mean last-3 val AUC-PR): {:.4f}'.format(mean_last3))

    # Сохраняю чекпоинт
    ckpt_filename = 'block2_rank{}_trial{}.pth'.format(rank_label, trial.number)
    ckpt_path     = os.path.join(OUTPUT_DIR, ckpt_filename)
    torch.save({
        'model_state_dict':     model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'params':               params,
        'val_auc_pr_history':   val_auc_pr_history,
        'epoch':                NUM_EPOCHS_BLOCK2,
        'trial_number':         trial.number,
        'rank':                 rank_label,
    }, ckpt_path)
    print('Веса сохранены:', ckpt_path)
    ipd.display(ipd.FileLink(ckpt_path))

    # Записываю результат в список
    def round_metrics(d):
        return {k: round(v, 6) if isinstance(v, float) else v for k, v in d.items()}

    block2_results.append({
        'trial_key':             trial_key,
        'rank':                  rank_label,
        'trial_number':          trial.number,
        'optuna_value':          round(trial.value, 6),
        'params':                params,
        'epoch_history':         epoch_history,
        'mean_last3_val_auc_pr': round(mean_last3, 6),
        'final_val_metrics':     round_metrics(final_val_metrics),
        'final_test_metrics':    round_metrics(final_test_metrics),
        'checkpoint_path':       ckpt_path,
    })

    # Сохраняю JSON сразу после каждой модели
    save_results(block2_results, block2_json_path)

    # Освобождаю GPU-память
    del model, optimizer, scheduler, criterion
    torch.cuda.empty_cache()


RANK 1 | Trial #6 | Optuna val_auc_pr=0.2679
Параметры: {'optimizer': 'Adam', 'scheduler': 'CosineAnnealingLR', 'lr': 1.2733617885633793e-05, 'weight_decay': 0.001393578530649149, 'dropout': 0.16977102154288687, 'label_smoothing': 0.11227438607129932}
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 208MB/s]


Ep  1/20 | loss=0.4072 | AUC-PR=0.2011 | F1=0.2476 | Rec=0.3562 | Prec=0.1898 | Spec=0.9727 | BalAcc=0.6644 | thr=0.75
Ep  2/20 | loss=0.2959 | AUC-PR=0.2396 | F1=0.3086 | Rec=0.3425 | Prec=0.2809 | Spec=0.9843 | BalAcc=0.6634 | thr=0.76
Ep  3/20 | loss=0.2716 | AUC-PR=0.2105 | F1=0.3158 | Rec=0.2466 | Prec=0.4390 | Spec=0.9943 | BalAcc=0.6205 | thr=0.77 | sliding3=0.2171
Ep  4/20 | loss=0.2585 | AUC-PR=0.1927 | F1=0.3151 | Rec=0.3151 | Prec=0.3151 | Spec=0.9877 | BalAcc=0.6514 | thr=0.53 | sliding3=0.2143
Ep  5/20 | loss=0.2493 | AUC-PR=0.2394 | F1=0.3529 | Rec=0.3288 | Prec=0.3810 | Spec=0.9904 | BalAcc=0.6596 | thr=0.48 | sliding3=0.2142
Ep  6/20 | loss=0.2455 | AUC-PR=0.2197 | F1=0.3359 | Rec=0.3014 | Prec=0.3793 | Spec=0.9912 | BalAcc=0.6463 | thr=0.49 | sliding3=0.2172
Ep  7/20 | loss=0.2407 | AUC-PR=0.2363 | F1=0.3356 | Rec=0.3425 | Prec=0.3289 | Spec=0.9875 | BalAcc=0.6650 | thr=0.39 | sliding3=0.2318
Ep  8/20 | loss=0.2382 | AUC-PR=0.2489 | F1=0.3270 | Rec=0.3562 | Prec=0.3023

/kaggle/working/block2_rank1_trial6.pth

Сохранено: /kaggle/working/block2_results.json


...

/kaggle/working/block2_results.json


RANK 3 | Trial #9 | Optuna val_auc_pr=0.2577
Параметры: {'optimizer': 'SGD', 'scheduler': 'CosineAnnealingLR', 'lr': 0.0016038791115402966, 'weight_decay': 0.0003882387971540929, 'dropout': 0.27270717062691385, 'label_smoothing': 0.029551498175417336}
Ep  1/20 | loss=0.3458 | AUC-PR=0.2100 | F1=0.2525 | Rec=0.3425 | Prec=0.2000 | Spec=0.9754 | BalAcc=0.6589 | thr=0.79
Ep  2/20 | loss=0.2023 | AUC-PR=0.1534 | F1=0.2534 | Rec=0.3836 | Prec=0.1892 | Spec=0.9705 | BalAcc=0.6770 | thr=0.48
Ep  3/20 | loss=0.1534 | AUC-PR=0.1873 | F1=0.2604 | Rec=0.3425 | Prec=0.2101 | Spec=0.9769 | BalAcc=0.6597 | thr=0.69 | sliding3=0.1836
Ep  4/20 | loss=0.1261 | AUC-PR=0.2225 | F1=0.3077 | Rec=0.2466 | Prec=0.4091 | Spec=0.9936 | BalAcc=0.6201 | thr=0.65 | sliding3=0.1877
Ep  5/20 | loss=0.1124 | AUC-PR=0.1967 | F1=0.2929 | Rec=0.3973 | Prec=0.2320 | Spec=0.9764 | BalAcc=0.6868 | thr=0.22 | sliding3=0.2022
Ep  6/20 | loss=0.1017 | AUC-PR=0.2493 | F1=0.3485 | Rec=0.3151 | Prec=0.3898 | Spec=0.9912 | BalA

/kaggle/working/block2_rank3_trial9.pth

Сохранено: /kaggle/working/block2_results.json


/kaggle/working/block2_results.json

In [28]:
# Итоговая таблица сравнения трёх моделей
print('\n' + '='*80)
print('ИТОГИ ЧАСТИ 5: СРАВНЕНИЕ ТОП-3 КОНФИГУРАЦИЙ')
print('='*80)

header = '{:<5} {:<10} {:<20} {:<12} {:<12} {:<12} {:<10}'.format(
    'Rank', 'Trial#', 'mean_last3_AUC-PR', 'Val AUC-PR', 'Test AUC-PR',
    'Test Recall', 'Test F1'
)
print(header)
print('-'*80)

for r in sorted(block2_results, key=lambda x: x['mean_last3_val_auc_pr'], reverse=True):
    print('{:<5} {:<10} {:<20.4f} {:<12.4f} {:<12.4f} {:<12.4f} {:<10.4f}'.format(
        r['rank'],
        r['trial_number'],
        r['mean_last3_val_auc_pr'],
        r['final_val_metrics']['auc_pr'],
        r['final_test_metrics']['auc_pr'],
        r['final_test_metrics']['recall'],
        r['final_test_metrics']['f1'],
    ))

best_b2 = max(block2_results, key=lambda r: r['mean_last3_val_auc_pr'])
print('\nЛучший по mean_last3_val_auc_pr:')
print('  Rank={} | Trial={} | mean_last3={:.4f} | Параметры: {}'.format(
    best_b2['rank'], best_b2['trial_number'],
    best_b2['mean_last3_val_auc_pr'], best_b2['params']))


ИТОГИ ЧАСТИ 5: СРАВНЕНИЕ ТОП-3 КОНФИГУРАЦИЙ
Rank  Trial#     mean_last3_AUC-PR    Val AUC-PR   Test AUC-PR  Test Recall  Test F1   
--------------------------------------------------------------------------------
3     9          0.2407               0.2210       0.2483       0.3151       0.3333    
1     6          0.2177               0.2065       0.2206       0.2877       0.3206    
2     15         0.2137               0.1555       0.2007       0.2466       0.2769    

Лучший по mean_last3_val_auc_pr:
  Rank=3 | Trial=9 | mean_last3=0.2407 | Параметры: {'optimizer': 'SGD', 'scheduler': 'CosineAnnealingLR', 'lr': 0.0016038791115402966, 'weight_decay': 0.0003882387971540929, 'dropout': 0.27270717062691385, 'label_smoothing': 0.029551498175417336}


---
## 6. Дообучение лучшей модели

Беру лучшую из топ-3 по **среднему val AUC-PR за последние 3 эпохи** из прошлой части. Загружаю её веса, optimizer и scheduler из чекпоинта части 5.  

**Критерий сохранения лучших весов:** максимальное скользящее среднее val AUC-PR за последние 3 эпохи из общей истории.  

**Early stopping:** patience=7 эпох без улучшения скользящего среднего. Счётчик ведётся от лучшего значения за **всё время** (включая предыдщую часть).  

**Первые 2 эпохи части 6 (глобально 21–22):** не сохраняю и не считаю patience. С 23-ей эпохи скользящее окно `[-3:]` заполнено только новыми эпохами.   

**Итоговая метрика для отчёта:** среднее val AUC-PR за последние 3 эпохи.

In [30]:
# Выбираю лучшую модель из топ-3 по среднему AUC-PR
best_b2 = max(block2_results, key=lambda r: r['mean_last3_val_auc_pr'])
print('Лучшая модель из части 5:')
print('  Rank={} | Trial={} | mean_last3_val_auc_pr={:.4f}'.format(
    best_b2['rank'], best_b2['trial_number'], best_b2['mean_last3_val_auc_pr']))
print('  Параметры: {}'.format(best_b2['params']))
print('  Загружаю из: {}'.format(best_b2['checkpoint_path']))

checkpoint = torch.load(best_b2['checkpoint_path'], map_location=DEVICE)
params     = checkpoint['params']

# Восстанавливаю модель с теми же параметрами, что в части 5
model = make_resnet50(dropout=params['dropout']).to(DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])

# Восстанавливаю оптимизатор
optimizer = make_optimizer(params, model)
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

# Перемещаю оптимизатор на нужное устройство
for state in optimizer.state.values():
    for k, v in state.items():
        if isinstance(v, torch.Tensor):
            state[k] = v.to(DEVICE)

# Восстанавливаю scheduler.
# Так как выиграл CosineAnnealingLR, то продолжаю с того места, где остановились
scheduler = make_scheduler(params, optimizer, NUM_EPOCHS_BLOCK3, len(train_loader))
scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
print('Scheduler {} загружен из checkpoint.'.format(params['scheduler']))

criterion = LabelSmoothingBCELoss(label_smoothing=params['label_smoothing'])

# Инициализируем early stopping
# История AUC-PR из части 5 нужна, чтобы скользящее окно корректно работало
val_auc_pr_history = checkpoint['val_auc_pr_history']
print('Загружена история части 5, ({} эпох).'.format(len(val_auc_pr_history)))

# Начальное лучшее скользящее среднее из конца части 5
best_sliding_mean = float(np.mean(val_auc_pr_history[-3:]))
best_weights      = copy.deepcopy(model.state_dict())
patience_counter  = 0
best_epoch_global = NUM_EPOCHS_BLOCK2

print('Начальное лучшее скользящее среднее (конец части 5): {:.4f}'.format(best_sliding_mean))
print('Дообучаю {} эпох с early stopping (patience={}).\n'.format(
    NUM_EPOCHS_BLOCK3, EARLY_STOPPING_PATIENCE))

block3_epoch_history = []
early_stopped        = False

scaler = torch.cuda.amp.GradScaler()

for epoch in range(NUM_EPOCHS_BLOCK3):
    epoch_local  = epoch + 1
    epoch_global = NUM_EPOCHS_BLOCK2 + epoch + 1

    # Обучаю одну эпоху
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scheduler, scaler=scaler)

    # Все метрики на val
    val_metrics = compute_metrics(model, val_loader, threshold=None)
    val_auc_pr_history.append(val_metrics['auc_pr'])

    # Скользящее среднее за крайние 3 эпохи
    sliding_mean = float(np.mean(val_auc_pr_history[-3:]))

    # Звёздочкой отмечаю лучшее значение
    star = ''
    if epoch >= 2 and sliding_mean > best_sliding_mean:
        star = ' BEST'

    print(
        'Ep {:2d}/30 (glob {:2d}) | loss={:.4f} | AUC-PR={:.4f} | F1={:.4f} | '
        'Rec={:.4f} | Prec={:.4f} | Spec={:.4f} | BalAcc={:.4f} | '
        'thr={:.2f} | AVG_3 AUC-PR={:.4f}{}'.format(
            epoch_local, epoch_global,
            train_loss,
            val_metrics['auc_pr'],
            val_metrics['f1'],
            val_metrics['recall'],
            val_metrics['precision'],
            val_metrics['specificity'],
            val_metrics['balanced_accuracy'],
            val_metrics['threshold'],
            sliding_mean,
            star,
        )
    )

    block3_epoch_history.append({
        'epoch_local':       epoch_local,
        'epoch_global':      epoch_global,
        'train_loss':        round(train_loss, 6),
        'val_auc_pr':        round(val_metrics['auc_pr'], 6),
        'val_f1':            round(val_metrics['f1'], 6),
        'val_recall':        round(val_metrics['recall'], 6),
        'val_precision':     round(val_metrics['precision'], 6),
        'val_specificity':   round(val_metrics['specificity'], 6),
        'val_balanced_acc':  round(val_metrics['balanced_accuracy'], 6),
        'val_threshold':     round(val_metrics['threshold'], 4),
        'sliding_mean_3':    round(sliding_mean, 6),
    })

    # Early stopping
    if epoch >= 2:
        if sliding_mean > best_sliding_mean:
            # Нашла лучшее скользящее среднее - сохраняю веса, сбрасываю терпение
            best_sliding_mean  = sliding_mean
            best_weights       = copy.deepcopy(model.state_dict())
            best_epoch_global  = epoch_global
            patience_counter   = 0
        else:
            patience_counter += 1

        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print('\nEarly stopping: {} эпох без улучшения.'.format(
                EARLY_STOPPING_PATIENCE))
            print('Лучшие веса: глобальная эпоха {}.'.format(best_epoch_global))
            early_stopped = True
            break

# Загружаю лучшие веса
model.load_state_dict(best_weights)
print('\nЗагружены лучшие веса (глобальная эпоха {}).'.format(best_epoch_global))
print('Лучшее скользящее среднее val AUC-PR: {:.4f}'.format(best_sliding_mean))

Лучшая модель из части 5:
  Rank=3 | Trial=9 | mean_last3_val_auc_pr=0.2407
  Параметры: {'optimizer': 'SGD', 'scheduler': 'CosineAnnealingLR', 'lr': 0.0016038791115402966, 'weight_decay': 0.0003882387971540929, 'dropout': 0.27270717062691385, 'label_smoothing': 0.029551498175417336}
  Загружаю из: /kaggle/working/block2_rank3_trial9.pth
Scheduler CosineAnnealingLR загружен из checkpoint.
Загружена история части 5, (20 эпох).
Начальное лучшее скользящее среднее (конец части 5): 0.2407
Дообучаю 30 эпох с early stopping (patience=7).

Ep  1/30 (glob 21) | loss=0.0916 | AUC-PR=0.2391 | F1=0.3145 | Rec=0.3425 | Prec=0.2907 | Spec=0.9850 | BalAcc=0.6637 | thr=0.15 | AVG_3 AUC-PR=0.2325
Ep  2/30 (glob 22) | loss=0.0910 | AUC-PR=0.2340 | F1=0.3025 | Rec=0.2466 | Prec=0.3913 | Spec=0.9931 | BalAcc=0.6198 | thr=0.57 | AVG_3 AUC-PR=0.2313
Ep  3/30 (glob 23) | loss=0.0892 | AUC-PR=0.2269 | F1=0.3175 | Rec=0.2740 | Prec=0.3774 | Spec=0.9919 | BalAcc=0.6329 | thr=0.33 | AVG_3 AUC-PR=0.2333
Ep  4/30

In [31]:
# Финальные метрики и сохранение результатов

# Порог подбираю на val с лучшими весами, применяем на test
final_val_metrics  = compute_metrics(model, val_loader,  threshold=None)
final_test_metrics = compute_metrics(model, test_loader,
                                     threshold=final_val_metrics['threshold'])

# Итоговая метрика для отчёта: среднее AUC-PR за последние 3 эпохи
last3_auc_pr      = [r['val_auc_pr'] for r in block3_epoch_history[-3:]]
mean_last3_block3 = float(np.mean(last3_auc_pr))

print('\n' + '='*70)
print('ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ ЧАСТИ 6')
print('='*70)
print('Эпох дообучения: {} из {} | Early stopping: {}'.format(
    len(block3_epoch_history), NUM_EPOCHS_BLOCK3, early_stopped))
print('Лучшие веса: глобальная эпоха {}'.format(best_epoch_global))
print('Лучшее скользящее среднее val AUC-PR: {:.4f}'.format(best_sliding_mean))
print('Итоговая метрика (mean last-3 val AUC-PR ЧАСТИ 6): {:.4f}'.format(mean_last3_block3))

print('\nVal-метрики (порог={:.2f}):'.format(final_val_metrics['threshold']))
print('  AUC-PR={:.4f} | F1={:.4f} | Recall={:.4f} | Precision={:.4f} | '
      'Specificity={:.4f} | BalAcc={:.4f}'.format(
    final_val_metrics['auc_pr'],  final_val_metrics['f1'],
    final_val_metrics['recall'],  final_val_metrics['precision'],
    final_val_metrics['specificity'], final_val_metrics['balanced_accuracy']))

print('\nTest-метрики (порог с val={:.2f}):'.format(final_val_metrics['threshold']))
print('  AUC-PR={:.4f} | F1={:.4f} | Recall={:.4f} | Precision={:.4f} | '
      'Specificity={:.4f} | BalAcc={:.4f}'.format(
    final_test_metrics['auc_pr'],  final_test_metrics['f1'],
    final_test_metrics['recall'],  final_test_metrics['precision'],
    final_test_metrics['specificity'], final_test_metrics['balanced_accuracy']))

# Сохраняю веса финальной модели
final_model_path = os.path.join(OUTPUT_DIR, 'block3_final_model.pth')
torch.save({
    'model_state_dict':           best_weights,
    'params':                     params,
    'best_epoch_global':          best_epoch_global,
    'best_sliding_mean':          best_sliding_mean,
    'val_auc_pr_history_full':    val_auc_pr_history,
    'early_stopped':              early_stopped,
}, final_model_path)
print('\nВеса финальной модели сохранены:', final_model_path)
ipd.display(ipd.FileLink(final_model_path))

# Сохраняю результаты в JSON
def round_metrics(d):
    return {k: round(v, 6) if isinstance(v, float) else v for k, v in d.items()}

block3_results = [{
    'source_trial_number':          best_b2['trial_number'],
    'source_rank':                  best_b2['rank'],
    'params':                       params,
    'epochs_run_block3':            len(block3_epoch_history),
    'early_stopped':                early_stopped,
    'best_epoch_global':            best_epoch_global,
    'best_sliding_mean_val_auc_pr': round(best_sliding_mean, 6),
    'mean_last3_val_auc_pr_block3': round(mean_last3_block3, 6),
    'epoch_history_block3':         block3_epoch_history,
    'final_val_metrics':            round_metrics(final_val_metrics),
    'final_test_metrics':           round_metrics(final_test_metrics),
    'model_path':                   final_model_path,
}]

block3_json_path = os.path.join(OUTPUT_DIR, 'block3_results.json')
save_results(block3_results, block3_json_path)

print('\nФинальная модель обучена и сохранена.')


ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ ЧАСТИ 6
Эпох дообучения: 9 из 30 | Early stopping: True
Лучшие веса: глобальная эпоха 20
Лучшее скользящее среднее val AUC-PR: 0.2407
Итоговая метрика (mean last-3 val AUC-PR ЧАСТИ 6): 0.2280

Val-метрики (порог=0.34):
  AUC-PR=0.2210 | F1=0.3385 | Recall=0.3014 | Precision=0.3860 | Specificity=0.9914 | BalAcc=0.6464

Test-метрики (порог с val=0.34):
  AUC-PR=0.2483 | F1=0.3333 | Recall=0.3151 | Precision=0.3538 | Specificity=0.9897 | BalAcc=0.6524

Веса финальной модели сохранены: /kaggle/working/block3_final_model.pth


/kaggle/working/block3_final_model.pth

Сохранено: /kaggle/working/block3_results.json


/kaggle/working/block3_results.json


Финальная модель обучена и сохранена.


## Результаты обучения ResNet-50: итоговый анализ

### Итоговые метрики

Финальная модель - ResNet-50 с полным fine-tuning на датасете ISIC 2020 - была обучена в 3 этапа: поиск гиперпараметров через Optuna (20 trials по 10 эпох), дообучение топ-3 конфигураций по 20 эпох, и финальное дообучение лучшей модели ещё до 50 эпох с early stopping. Лучшая конфигурация (SGD + momentum + CosineAnnealingLR) остановилась на эпохе 20 - дообучение в части 6 не принесло улучшений, early stopping сработал на 29-ой эпохе из 50.

Финальные результаты на тестовой выборке:

- **AUC-PR = 0.2483** - главная метрика; показывает качество ранжирования при сильном дисбалансе классов, не зависит от порога бинаризации
- **Recall = 0.3151** - доля найденных меланом среди всех реальных меланом в тесте
- **Precision = 0.3538** - доля реальных меланом среди всех случаев, которые модель назвала меланомой
- **F1 = 0.3333** - среднее гармоническое Recall и Precision; балансирует между ними
- **Specificity = 0.9897** - доля верно классифицированных здоровых случаев
- **Balanced Accuracy = 0.6524** - среднее между Recall и Specificity; корректная accuracy при дисбалансе

### Интерпретация метрик

**AUC-PR в 0.25** нужно оценивать относительно базового уровня. При доле положительного класса менее 2% случайный классификатор даёт AUC-PR = 0.02. Полученное значение превышает случайный уровень примерно в 12 раз, что свидетельствует о том, что модель действительно обнаружила информативный сигнал в данных.

**Recall 0.32** означает, что модель пропускает около 68% меланом. **Precision 0.35** при этом говорит о том, что чуть больше трети предсказаний «меланома» действительно оказываются меланомой, а остальное ложные тревоги. **F1 0.33** отражает этот компромисс: модель одинаково плохо и находит меланомы, и уверена в своих предсказаниях. В клинической практике приоритет обычно отдаётся Recall - пропустить меланому опаснее, чем назначить лишнее обследование, — поэтому в следующих итерациях имеет смысл сместить порог бинаризации в сторону меньших значений.

**Specificity 0.99 и Balanced Accuracy 0.65** вместе показывают асимметрию модели: она отлично распознаёт здоровых (Specificity почти идеальный), но слабо справляется с меланомами. Это типичное поведение при сильном дисбалансе даже при использовании WeightedRandomSampler - сэмплер балансирует батчи во время обучения, но не устраняет фундаментальную сложность задачи.

### Возможные причины ограниченного качества

**Предобученные веса ImageNet.** Использование весов ImageNet теоретически оправдано - низкоуровневые детекторы краёв и текстур универсальны. Однако дерматоскопические снимки существенно отличаются от фотографий из ImageNet: специфические паттерны меланомы (неравномерная пигментация, атипичная сосудистая сеть, структура поверхности кожи) не встречаются в ImageNet. Это могло ограничить качество признаков, которые модель научилась извлекать.

**Малое число эпох в Optuna.** Каждый trial обучался всего 10 эпох, тогда как полноценное обучение проходило на 20–50. Это означает, что Optuna оценивала конфигурации в условиях, далёких от финального качества, и могла отдать предпочтение конфигурациям, которые быстро сходятся, а не тем, которые в итоге дают лучший результат.

**Ранняя сходимость.** Модель перестала улучшаться уже к 20-й эпохе. Это может указывать на то, что выбранный learning rate (0.0016 для SGD) слишком велик для
тонкой настройки на поздних эпохах, либо на то, что CosineAnnealingLR с T_max=10 завершил цикл и lr вернулся к высокому значению в момент, когда модели нужен был меньший шаг.

**Дисбаланс классов.** Несмотря на WeightedRandomSampler, дисбаланс 1:49 крайне сложен. Сэмплер балансирует батчи, но не меняет то, что меланомы принципиально сложнее для обучения из-за малого числа уникальных примеров и высокой визуальной схожести с доброкачественными образованиями.